# Propensity: who's likely to respond

A classifier that gives each customer the chance they'll respond to a campaign (or buy a product, or upgrade), learned from who responded last time: FLAML's AutoML picks the model, it's logged and registered in the project's own MLflow workspace, and once another editor approves it, the project's scoring scores every customer on a schedule, so the next campaign goes to the likeliest.

**To use it:** save a copy (File → Save Notebook As…), then change the parameters in the next cell: the project, the features table and its id column, the query that says who responded, and the model's name. The examples are the guide's `retail` project (who converted on the September clearance, from what was known on the day it started); run top to bottom. Data comes through the platform's SQL, as you, at most 100,000 rows: aggregate in SQL, a row per customer.

In [ ]:
PROJECT = "retail"                 # the project: its tables, its MLflow workspace, its Models tab
FEATURES = "customer_campaign_features"   # the model's inputs, a row per ID: the table the scoring will score
ID = "customer_id"                 # the column naming each row (the scores carry it, so they join back)
COLUMNS = None                     # the inputs, by name; None: every column of FEATURES but ID
LABEL = "converted"                # 1: they responded (here: bought through campaign 6); 0: they didn't. From LABELS:
LABELS = """
SELECT c.customer_id,
       CASE WHEN count_if(r.converted) > 0 THEN 1 ELSE 0 END AS converted
FROM uploads_customers c
LEFT JOIN uploads_campaign_responses r ON r.customer_id = c.customer_id AND r.campaign_id = 6
GROUP BY c.customer_id
"""
MODEL = "propensity"               # the registered model's name
EXPERIMENT = "propensity"          # the MLflow experiment its runs go to
TIME_BUDGET = 60                   # seconds FLAML may spend trying models

## The features

**One table, two uses.** A scoring scores a table with the model, so the model learns from one: a row per `ID`, its other columns the model's inputs, as numbers (integers, or `double`: CAST decimals; a 0/1 column per value of a text column), and no label. The label comes from its own query, joined on `ID`. The inputs describe each row **before** the period the label looks at (the cutoff), or the model learns from the answer.

Make the table once, as an editor: adapt the SQL below to your data and run it here (uncomment the last line) or on the project's **SQL** tab.

In [ ]:
FEATURES_SQL = f"""
CREATE OR REPLACE TABLE {FEATURES} AS
WITH o AS (
  SELECT customer_id, count(*) AS orders, CAST(sum(amount) AS double) AS revenue, max(ordered_on) AS last_order
  FROM uploads_orders WHERE NOT test AND NOT refunded AND ordered_on < DATE '2026-09-05'
  GROUP BY customer_id),
r AS (
  SELECT customer_id, count(*) AS responses, count_if(converted) AS conversions
  FROM uploads_campaign_responses WHERE responded_on < DATE '2026-09-05'
  GROUP BY customer_id)
SELECT c.customer_id,
       coalesce(o.orders, 0) AS orders,
       coalesce(o.revenue, 0.0) AS revenue,
       coalesce(date_diff('day', o.last_order, DATE '2026-09-05'), 9999) AS days_since_order,
       coalesce(r.responses, 0) AS responses,
       coalesce(r.conversions, 0) AS conversions,
       date_diff('day', c.signed_up, DATE '2026-09-05') AS tenure_days,
       CASE WHEN c.segment = 'small business' THEN 1 ELSE 0 END AS small_business,
       CASE WHEN c.segment = 'enterprise' THEN 1 ELSE 0 END AS enterprise
FROM uploads_customers c
LEFT JOIN o ON o.customer_id = c.customer_id
LEFT JOIN r ON r.customer_id = c.customer_id
"""
# import buckets_lake; buckets_lake.sql(PROJECT, FEATURES_SQL)

## The data

The features and the label, joined on `ID`. Few responders make a noisy model: hundreds of rows with a few dozen 1s at least.

In [ ]:
import buckets_lake
import pandas as pd

features = buckets_lake.sql(PROJECT, f"SELECT * FROM {FEATURES}")   # as you, under the project's rules
labels = buckets_lake.sql(PROJECT, LABELS)
data = features.merge(labels[[ID, LABEL]], on=ID)                   # the rows with both
columns = COLUMNS or [c for c in features.columns if c != ID]
text = [c for c in columns if not pd.api.types.is_numeric_dtype(data[c])]
assert not text, f"the model takes numbers: in {FEATURES}, CAST these or make a 0/1 column per value: {text}"
X, y = data[columns], data[LABEL].astype("int64")
print(f"{len(data)} rows, {y.mean():.0%} with {LABEL} = 1; inputs: {', '.join(columns)}")
X.describe()

## Train, log and register

FLAML picks the model by ROC AUC (how well it ranks who responded above who didn't). The run, its parameters and metrics, and the model are kept in the project's MLflow workspace; `registered_model_name` makes it a new version of `MODEL`.

In [ ]:
import skops.io
from flaml import AutoML
from sklearn.base import clone
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split

# A quarter held out, to say how well the model ranks rows it hasn't seen.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, stratify=y, random_state=0)

mlflow = buckets_lake.mlflow(PROJECT)        # the project's own MLflow workspace, as you
mlflow.set_experiment(EXPERIMENT)
with mlflow.start_run():                     # stamped with you and the catalog's commit
    automl = AutoML()
    # FLAML tries models and settings for TIME_BUDGET seconds. Only the kinds a scoring can
    # load (scikit-learn's, LightGBM's, XGBoost's): the scoring has those, not FLAML.
    automl.fit(X_train, y_train, task="classification", metric="roc_auc", time_budget=TIME_BUDGET,
               estimator_list=["lgbm", "xgb_limitdepth", "rf", "extra_tree", "lrl1"], seed=0, verbose=0,
               mlflow_logging=False)
    best = automl.model.estimator            # the best model itself, without FLAML around it
    auc = roc_auc_score(y_test, best.predict_proba(X_test)[:, 1])
    model = clone(best).fit(X, y)            # the same model and settings, learned again from every row
    mlflow.log_params({"table": FEATURES, "inputs": ",".join(columns), "rows": len(X), "estimator": automl.best_estimator,
                       **{f"best_{k}": v for k, v in automl.best_config.items()}})
    mlflow.log_metrics({"roc_auc": auc, "roc_auc_validation": 1 - automl.best_loss, "positive_rate": float(y.mean())})
    mlflow.sklearn.log_model(model, name="model", registered_model_name=MODEL,
                             pyfunc_predict_fn="predict_proba",   # a scoring takes the last column: the chance of 1
                             input_example=X.head(),              # the signature: the columns, int64 and float64
                             skops_trusted_types=skops.io.get_untrusted_types(data=skops.io.dumps(model)))
print(f"{automl.best_estimator}: ROC AUC {auc:.3f} on the held-out quarter (0.5 is a coin, 1 is perfect)")

The customers likeliest to respond, by the model: the top of the next campaign's list.

In [ ]:
data.assign(score=model.predict_proba(X)[:, 1]).sort_values("score", ascending=False).head(10)

## Next

1. **Its card, and an approval.** On the project's **Models** tab, `propensity` has a new version, registered by you. **Write its card** (what it's for, the data it learned from, its metrics, its limits), and have another editor of the project approve the version: nobody approves their own. The alias `approved` then names it, and that's the version a scoring loads (`models:/propensity@approved`).
2. **Score the table on a schedule.** In the project's spec (**Edit** → JSON), add a scoring:

   ```json
   "scoring": [{"name": "propensity-nightly", "model": "propensity", "table": "customer_campaign_features", "id": "customer_id",
                "output": "customer_propensity_scores", "schedule": "0 4 * * 1", "drift": 0.2}]
   ```

   If you set `COLUMNS`, add `"columns": [...]` with the same names. Within a minute the **Pipelines** tab lists the scoring; each run, as the project's pipelines, writes `customer_propensity_scores` afresh: each row's `customer_id`, its score (the chance of 1), the model's version and when. When an input column drifts past `drift` from the version's first run, the project's alerts say so.
3. **Keep the features current.** A scoring scores the table as it is: rebuild `customer_campaign_features` with today as the cutoff (the same SQL, as one of the project's pipelines) before it runs.
4. **Retrain** when the alerts say an input has drifted, or the scores stop matching what happens: run this notebook again, and have the new version approved. Its first scoring run becomes the new reference.